# Save and load a master pattern

This notebook shows the full round-trip:

1. Simulate a GaN master pattern.
2. Save it to a compressed `.npz` (fundamental sector + intermediates + metadata).
3. Reload it with the NumPy-only loader; the full Lambert hemispheres are expanded on load into `mp.data`.
4. Export PNGs of the integrated pattern and of each per-energy-bin intermediate.

The loader (`ebsdsim.mploader`) only needs NumPy, so another project can copy that one file and read these `.npz` files without installing `ebsdsim` or having a GPU.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

import ebsdsim as es
from ebsdsim.mploader import load_master_pattern, save_png_gray, to_uint8

out_dir = Path("_mp_out")
out_dir.mkdir(exist_ok=True)

## 1. Simulate and save

GaN has two sites (Ga, N) and is non-centrosymmetric, so the south hemisphere is genuinely present.

In [ ]:
gan = es.Material(
    cell=es.Cell(a=3.18893, b=3.18893, c=5.19236, gamma=120.0, space_group=186),
    atoms=[
        es.Atom("Ga", x=1 / 3, y=2 / 3, z=0.99908, b_iso=0.5),
        es.Atom("N", x=1 / 3, y=2 / 3, z=0.37592, b_iso=0.5),
    ],
    name="GaN",
)
mp = es.master_pattern(gan, voltage_kv=20.0, halfw=250)

npz_path = mp.save(out_dir / "GaN-master-pattern.npz")
print("saved:", npz_path, "(", npz_path.stat().st_size, "bytes )")
print("intermediate bins saved:", len(mp.bin_patterns))

## 2. Reload with the NumPy-only loader

In [ ]:
loaded = load_master_pattern(npz_path)
print("format          :", loaded.meta["format"])
print("side            :", loaded.side)
print("n_sites         :", loaded.n_sites)
print("n_bins          :", loaded.n_bins)
print("needs southern  :", loaded.needs_southern_hemisphere)
print("centrosymmetric :", loaded.is_centrosymmetric)

## 3. The expanded data tensor

`load_master_pattern` expands the fundamental sector on load into `mp.data`, a
`(energy, site, hemisphere, H, W)` array:

- energy axis: index 0 is the energy-integrated pattern, `1..` are the bins (only when there is more than one bin).
- site axis: index 0 is the site-integrated pattern, `1..` are the individual sites (only when there is more than one site).
- hemisphere axis: index 0 is north, index 1 is south (present only when needed).

So `mp.data[0, 0, 0]` is the conventional energy- and site-integrated north
hemisphere, and it matches the GPU output to float32 precision.

In [ ]:
import numpy as np

print("data shape:", loaded.data.shape, "  axes:", loaded.axes["dims"])

nh = loaded.data[0, 0, 0]
print("loader-vs-GPU NH max abs diff:", float(np.max(np.abs(nh - mp.pattern))))

if loaded.needs_southern_hemisphere:
    sh = loaded.data[0, 0, 1]
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    axes[0].imshow(nh, cmap="gray"); axes[0].set_title("Integrated NH"); axes[0].axis("off")
    axes[1].imshow(sh, cmap="gray"); axes[1].set_title("Integrated SH"); axes[1].axis("off")
    plt.tight_layout(); plt.show()
else:
    plt.figure(figsize=(4.5, 4.5))
    plt.imshow(nh, cmap="gray"); plt.title("Integrated NH"); plt.axis("off")
    plt.show()

## 4. Export PNGs of the integrated pattern and each intermediate

In [ ]:
save_png_gray(to_uint8(loaded.data[0, 0, 0]), out_dir / "GaN_integrated_nh.png")
if loaded.needs_southern_hemisphere:
    save_png_gray(to_uint8(loaded.data[0, 0, 1]), out_dir / "GaN_integrated_sh.png")

for b in range(loaded.n_bins):
    e = loaded.axes["bin_to_energy_index"][b]
    voltage = float(loaded.bin_voltages_kv[b])
    save_png_gray(to_uint8(loaded.data[e, 0, 0]), out_dir / f"GaN_bin{b:02d}_{voltage:.1f}kV_nh.png")

print("wrote:")
for p in sorted(out_dir.glob("*.png")):
    print("  ", p.name)

## 5. Embedded metadata

Two `.npz` files differ in metadata whenever any pattern-affecting parameter changes — including the per-site Debye–Waller factors.

In [ ]:
m = loaded.meta
print("rank          :", m["rank"])
print("bethe strong  :", m["bethe_c_strong"])
print("bethe weak    :", m["bethe_c_weak"])
print("bethe cutoff  :", m["bethe_c_cutoff"])
print("dmin          :", m["dmin"])
print()
for site in m["cell"]["sites"]:
    print(f"  {site['symbol']:>2}  B_iso = {site['b_iso_angstrom_sq']:.3f} Å²")